# Neural Network Predicting Galaxy Potentials / Density Profile

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Check torch is working and whether you have a GPU
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

## Phase 1: PINN sanity check on an analytic NFW profile

Goal: check that a neural network can learn to *solve* the Poisson equation

$$\nabla^2 \Phi = \frac{1}{r^2}\frac{d}{dr}\left(r^2 \frac{d\Phi}{dr}\right) = 4\pi G \rho(r)$$

for a spherical NFW density profile, using only the PDE itself as the training signal (no precomputed potential values as labels in the interior) — validated against the known closed-form NFW potential.

**NFW profile** (dimensionless units, $G = \rho_0 = r_s = 1$):

$$\rho(r) = \frac{\rho_0}{(r/r_s)(1+r/r_s)^2} \qquad\qquad \Phi(r) = -\frac{4\pi G \rho_0 r_s^3}{r}\ln\left(1+\frac{r}{r_s}\right)$$

**Boundary conditions.** NFW is cuspy ($\rho \sim r^{-1}$ as $r\to0$), so the enclosed mass goes as $M(r) \sim r^2$ near the center and $d\Phi/dr \to$ a finite *nonzero* constant at $r=0$ (not the "regularity" condition $d\Phi/dr=0$ you'd use for a cored profile like Plummer). Rather than rely on getting that inner asymptotic exactly right, we solve the ODE as a two-point boundary value problem on a finite shell $[r_\mathrm{min}, r_\mathrm{max}]$, using the known analytic $\Phi$ only to supply the two Dirichlet boundary values — standing in for "enclosed mass known at small/large radius," which is what you'd have from real data too.

In [ ]:
# Analytic NFW density / potential (dimensionless units: G = rho_0 = r_s = 1)
G, rho_0, r_s = 1.0, 1.0, 1.0

def nfw_density(r):
    x = r / r_s
    return rho_0 / (x * (1 + x)**2)

def nfw_potential(r):
    x = r / r_s
    return -4 * np.pi * G * rho_0 * r_s**3 * np.log(1 + x) / r

def nfw_enclosed_mass(r):
    x = r / r_s
    return 4 * np.pi * rho_0 * r_s**3 * (np.log(1 + x) - x / (1 + x))

r_min, r_max = 1e-2, 20.0  # domain for the BVP (in units of r_s)

r_plot = np.logspace(np.log10(r_min), np.log10(r_max), 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(r_plot, nfw_density(r_plot))
axes[0].set_xlabel(r'$r / r_s$'); axes[0].set_ylabel(r'$\rho(r)$'); axes[0].set_title('NFW density (analytic)')

axes[1].semilogx(r_plot, nfw_potential(r_plot))
axes[1].set_xlabel(r'$r / r_s$'); axes[1].set_ylabel(r'$\Phi(r)$'); axes[1].set_title('NFW potential (analytic)')
plt.tight_layout()
plt.show()

### Network and PDE residual

A small MLP maps $r \mapsto \Phi(r)$. `torch.autograd` gives us $d\Phi/dr$ and $d^2\Phi/dr^2$ directly from the network output, so the Poisson equation residual can be evaluated at any collocation point without ever needing a labelled $\Phi$ value in the interior.

One practical wrinkle: $\rho(r)$ spans ~4 orders of magnitude between $r_\mathrm{min}$ and $r_\mathrm{max}$, so an un-normalized residual $\left(d^2\Phi/dr^2 + \tfrac{2}{r}d\Phi/dr - 4\pi G\rho\right)^2$ is dominated by the small-$r$ points where $\rho$ is huge, starving the optimizer's attention on the outer region (confirmed empirically below — the raw residual converges the inner region fast but leaves ~25% relative error in the middle/outer shell). Dividing the residual by the local $4\pi G \rho(r)$ scale fixes this.

In [ ]:
torch.manual_seed(0)

class PotentialNet(nn.Module):
    """MLP mapping r -> Phi(r)."""
    def __init__(self, n_hidden=64, n_layers=3):
        super().__init__()
        layers = [nn.Linear(1, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        layers += [nn.Linear(n_hidden, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, r):
        return self.net(r)

# float64 throughout: second derivatives via autograd are sensitive to precision
model = PotentialNet().to(device).double()

def sample_collocation(n):
    """Log-spaced (jittered) collocation points in [r_min, r_max]."""
    log_r = np.random.uniform(np.log10(r_min), np.log10(r_max), n)
    r = 10**log_r
    return torch.tensor(r, dtype=torch.float64, device=device).unsqueeze(1)

def pde_residual(model, r):
    r = r.clone().requires_grad_(True)
    phi = model(r)
    dphi_dr = torch.autograd.grad(phi, r, grad_outputs=torch.ones_like(phi), create_graph=True)[0]
    d2phi_dr2 = torch.autograd.grad(dphi_dr, r, grad_outputs=torch.ones_like(dphi_dr), create_graph=True)[0]
    rho = nfw_density(r)
    # normalized so the huge small-r density doesn't dominate the loss (see markdown above)
    return (d2phi_dr2 + (2.0 / r) * dphi_dr - 4 * np.pi * G * rho) / (4 * np.pi * G * rho)

# Dirichlet boundary values from the known analytic solution
r_min_t = torch.tensor([[r_min]], dtype=torch.float64, device=device)
r_max_t = torch.tensor([[r_max]], dtype=torch.float64, device=device)
phi_min_target = torch.tensor([[nfw_potential(r_min)]], dtype=torch.float64, device=device)
phi_max_target = torch.tensor([[nfw_potential(r_max)]], dtype=torch.float64, device=device)

def bc_loss():
    return ((model(r_min_t) - phi_min_target)**2 + (model(r_max_t) - phi_max_target)**2).squeeze()

### Training: Adam warmup + L-BFGS fine-tune

Adam gets the loss down quickly but plateaus around $10^{-3}$–$10^{-4}$ and oscillates (typical for PINNs, since resampled collocation points each step act like noisy minibatches). A short L-BFGS pass on a fixed collocation set afterwards squeezes the residual down several more orders of magnitude — standard practice for PINNs once Adam has found the right basin.

In [ ]:
n_collocation = 2000
bc_weight = 10.0
n_epochs = 8000

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_history = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    r_colloc = sample_collocation(n_collocation)
    residual = pde_residual(model, r_colloc)
    loss_pde = torch.mean(residual**2)
    loss_bc = bc_loss()
    loss = loss_pde + bc_weight * loss_bc
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    if epoch % 1000 == 0:
        print(f"[adam]  epoch {epoch:5d}  loss {loss.item():.4e}  pde {loss_pde.item():.4e}  bc {loss_bc.item():.4e}")

# L-BFGS fine-tune on a fixed collocation set
r_colloc_fixed = sample_collocation(4000)
lbfgs = optim.LBFGS(model.parameters(), lr=0.5, max_iter=500, history_size=50,
                     tolerance_grad=1e-12, tolerance_change=1e-14, line_search_fn='strong_wolfe')

def closure():
    lbfgs.zero_grad()
    loss = torch.mean(pde_residual(model, r_colloc_fixed)**2) + bc_weight * bc_loss()
    loss.backward()
    return loss

lbfgs.step(closure)
print(f"[lbfgs] final loss {closure().item():.4e}")

In [ ]:
# Evaluate against the analytic NFW potential
r_test = np.logspace(np.log10(r_min), np.log10(r_max), 300)
r_test_t = torch.tensor(r_test, dtype=torch.float64, device=device).unsqueeze(1)
with torch.no_grad():
    phi_nn = model(r_test_t).squeeze().numpy()
phi_true = nfw_potential(r_test)
rel_err = np.abs((phi_nn - phi_true) / phi_true)

print(f"max relative error:  {rel_err.max():.3e}")
print(f"mean relative error: {rel_err.mean():.3e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(loss_history)
axes[0].set_yscale('log')
axes[0].set_xlabel('Adam epoch'); axes[0].set_ylabel('total loss'); axes[0].set_title('Training loss (Adam phase)')

axes[1].semilogx(r_test, phi_true, label='analytic', lw=2)
axes[1].semilogx(r_test, phi_nn, '--', label='PINN', lw=2)
axes[1].set_xlabel(r'$r / r_s$'); axes[1].set_ylabel(r'$\Phi(r)$'); axes[1].set_title('PINN vs analytic potential')
axes[1].legend()

axes[2].loglog(r_test, rel_err)
axes[2].set_xlabel(r'$r / r_s$'); axes[2].set_ylabel('relative error'); axes[2].set_title('PINN relative error')

plt.tight_layout()
plt.show()

**Result:** ~$10^{-6}$ relative error across the whole shell after the L-BFGS fine-tune — the network has learned to solve the Poisson equation for NFW to numerical precision, using only the PDE and two boundary values.

**Next steps** (not yet implemented):
- Condition the network on profile parameters (e.g. feed in `[r, M_vir, c]` or the fitted params from `ModelRegistry`) so *one* trained network generalizes across NFW halos of different mass/concentration, instead of retraining per halo.
- Extend to composite profiles (bulge + disk + DM halo) in cylindrical coordinates $(R, z)$, where no closed form exists — this is where the PINN approach actually earns its keep over the existing analytic mass integrators in `density_profiles.py`.
- If moving to real FOGGIE halos: swap the Dirichlet boundary values for the enclosed-mass-based Newtonian tail (outer) and the innermost resolved density-profile point (inner), rather than a closed-form solution.